# Parse instruments
Version the reference data carried by checked FIX market messages.


In [ ]:
project_root = "."
# Checked FIX market messages, as `parse_fix` wrote them.
source = "fix.market"
start = None
end = None
fix_dictionary = "data/fix"
catalog = "rekep"
catalog_properties = {}
table_properties = {"history.expire.max-snapshot-age-ms": "604800000"}
branch = "root"
target = "market.instruments"
batch_row_size = 65_536
commit_row_size = 250_000
log_level = "INFO"


In [ ]:
from pyiceberg.expressions import And, GreaterThanOrEqual, In, LessThan

from rekep.fix.registry import FixRegistry
from rekep.iceberg import IcebergDataset
from rekep.logs import configure
from rekep.market import Instrument, versioned
from rekep.text import FixMsg
from rekep.times import unix_of
from rekep.urls import Url

configure(log_level)


def _window(lower, upper, column="unix"):
    predicates = []
    if lower is not None:
        predicates.append(GreaterThanOrEqual(column, lower))
    if upper is not None:
        predicates.append(LessThan(column, upper))
    return None if not predicates else predicates[0] if len(predicates) == 1 else And(*predicates)


# The FIX stage resolved the transaction clock and wrote it as `unix`, which is
# what this table is sorted and partitioned on -- so the interval is read off
# the same column here, without asking the stage that filled it for its bounds.
lower, upper = unix_of(start), unix_of(end, upper=True)
registry = FixRegistry(
    cache_dir=Url.from_string(str(fix_dictionary)).resolve(project_root),
    announce=print,
)
field = FixMsg.into_field()
messages = IcebergDataset(
    field=field.with_name(source),
    catalog=catalog,
    properties=dict(catalog_properties),
    branch=branch,
)
instruments = IcebergDataset(
    field=Instrument.into_field(target),
    catalog=catalog,
    properties=dict(catalog_properties),
    table_properties=dict(table_properties),
    branch=branch,
)


In [ ]:
read = written = 0


def _observed():
    """Every instrument the window's messages describe, merged per ticker."""
    if not messages.exists:
        return iter(())
    reader = messages.read_arrow_reader(
        field, row_filter=_window(lower, upper), order_by=("unix", "msgseqnum", "hash")
    )
    return Instrument.from_fixmsgs(FixMsg.from_arrow_reader(reader), registry=registry)


def _stored(symboltickers):
    """What the table already holds for the tickers in one batch."""
    if not instruments.exists or not symboltickers:
        return {}
    reader = instruments.read_arrow_reader(
        Instrument.into_field(), row_filter=In("symbolticker", symboltickers)
    )
    return {row.symbolticker: row for row in Instrument.from_arrow_reader(reader)}


def _versions():
    """Only what changes the table, one bounded lookup per batch."""
    global read, written
    for batch in Instrument.into_arrow_reader(_observed(), batch_row_size=batch_row_size):
        observed = list(Instrument.from_arrow_reader(iter((batch,))))
        read += len(observed)
        changed = list(versioned(observed, _stored(tuple(row.symbolticker for row in observed))))
        if changed:
            written += len(changed)
            yield Instrument.into_arrow_batch(changed)


def _with_first(first, rest):
    yield first
    yield from rest


# `overwrite_arrow_reader` and not an append: a ticker holds one row, and a
# version replaces it. Nothing is written at all when no batch changed, so a
# replay of an unchanged window commits no snapshot.
changes = iter(_versions())
first = next(changes, None)
if first is not None:
    instruments.overwrite_arrow_reader(
        _with_first(first, changes),
        Instrument.into_field(),
        merge_by=True,
        commit_row_size=commit_row_size,
    )

result = {
    "read": read,
    "written": written,
    "skipped": read - written,
    "source": source,
    "target": instruments.name,
}
try:
    import scrapbook as sb
except ImportError:
    pass
else:
    sb.glue("result", result, encoder="json")
result
